# Population and Cancer Source Tables EDA
**DNA Gene Mapping Project - ML Phase V5**  
**Author:** Sharique Mohammad  
**Date:** February 2026  
**Tables covered:**
- cancer_variant_ml_features (3.1M variants, 27 cols)
- population_frequency_ml_features (46K variants, 28 cols)

## Objective
EDA on the cancer source table and population frequency source table. These are the original gold feature tables before enrichment, used for use cases 7 and 8.

## Use Cases Covered
- UC7: Cancer Variant Classification (target: is_driver_candidate)
- UC8: Population Carrier Screening (target: is_carrier_screening_candidate)
- UC15: Cancer Variant Molecular Classification (target: gene_cancer_role)

## Deliverables
- Visualizations saved per table under respective analytical folders
- EDA reports and metrics per table

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path().absolute().parent.parent
ANALYTICAL   = PROJECT_ROOT / 'data' / 'analytical'

def make_dirs(table_name):
    base = ANALYTICAL / table_name
    (base / 'images').mkdir(parents=True, exist_ok=True)
    (base / 'reports').mkdir(parents=True, exist_ok=True)
    (base / 'metrics').mkdir(parents=True, exist_ok=True)
    return base / 'images', base / 'reports', base / 'metrics'

print("Setup complete")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")
print(f"Host : {POSTGRES_HOST}:{POSTGRES_PORT}")
print(f"DB   : {POSTGRES_DB}")

---
## 3. cancer_variant_ml_features
**Use Cases 7 and 15**
**Targets:** is_driver_candidate (UC7), gene_cancer_role (UC15 multi-class)

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('cancer_variant_ml_features')

print("Loading cancer_variant_ml_features (10% sample)...")
df_cvc = pd.read_sql(
    "SELECT * FROM gold.cancer_variant_ml_features TABLESAMPLE SYSTEM (10)",
    engine
)
print(f"Rows: {len(df_cvc):,}  Cols: {len(df_cvc.columns)}")

int_cols = [
    'position', 'sample_count', 'total_mutation_count', 'missense_sample_count',
    'truncating_sample_count', 'silent_sample_count', 'snv_sample_count',
    'indel_sample_count', 'gene_total_samples', 'gene_unique_sites',
    'cancer_mutation_burden_score'
]
double_cols = ['cancer_priority_score']
bool_cols = [
    'is_recurrent_mutation', 'is_hotspot_mutation', 'is_high_impact_cancer_variant',
    'is_driver_candidate', 'is_cancer_gene', 'is_tumor_suppressor_candidate',
    'is_oncogene_candidate'
]

for col in int_cols:
    if col in df_cvc.columns:
        df_cvc[col] = pd.to_numeric(df_cvc[col], errors='coerce').astype('Int64')
for col in double_cols:
    if col in df_cvc.columns:
        df_cvc[col] = pd.to_numeric(df_cvc[col], errors='coerce')
for col in bool_cols:
    if col in df_cvc.columns:
        df_cvc[col] = df_cvc[col].astype(str).str.lower().map({'true': True, 'false': False})

total_cvc = len(df_cvc)

driver_cvc = int(df_cvc['is_driver_candidate'].sum()) if 'is_driver_candidate' in df_cvc.columns else 0
print(f"\nUC7 target is_driver_candidate : {driver_cvc:,} ({driver_cvc/total_cvc*100:.1f}%)")
if driver_cvc > 0 and (total_cvc - driver_cvc) > 0:
    ratio_cvc = max(driver_cvc, total_cvc-driver_cvc) / min(driver_cvc, total_cvc-driver_cvc)
    print(f"Imbalance : {ratio_cvc:.2f}:1  SMOTE: {ratio_cvc > 5}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['Driver', 'Non-Driver'], [driver_cvc, total_cvc - driver_cvc],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([driver_cvc, total_cvc - driver_cvc]):
    axes[0].text(i, val, f'{val:,}\n({val/total_cvc*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('UC7: Driver Candidate Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'gene_cancer_role' in df_cvc.columns:
    gcr_dist = df_cvc['gene_cancer_role'].value_counts()
    gcr_dist.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('UC15: Gene Cancer Role (multi-class target)', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variables.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variables.png")

In [ ]:
if 'mutation_frequency_category' in df_cvc.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    mfc = df_cvc['mutation_frequency_category'].value_counts()
    mfc.sort_values().plot(kind='barh', ax=axes[0], color='coral', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[0].set_title('Mutation Frequency Category', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    if 'cancer_priority_score' in df_cvc.columns:
        axes[1].hist(df_cvc['cancer_priority_score'].dropna(), bins=40,
                     color='darkred', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Cancer Priority Score', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
        axes[1].set_title('Cancer Priority Score Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02_mutation_frequency_priority.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 02_mutation_frequency_priority.png")

numeric_cvc = ['sample_count','total_mutation_count','missense_sample_count',
               'gene_total_samples','gene_unique_sites','cancer_mutation_burden_score',
               'cancer_priority_score']
numeric_cvc = [c for c in numeric_cvc if c in df_cvc.columns]

n_cols = 3
n_rows = (len(numeric_cvc) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()
for i, col in enumerate(numeric_cvc):
    data = df_cvc[col].dropna()
    axes[i].hist(data, bins=40, color='steelblue', alpha=0.8, edgecolor='black')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].grid(alpha=0.3)
for i in range(len(numeric_cvc), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '03_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_numeric_distributions.png")

miss_cvc = pd.DataFrame({'column': df_cvc.columns,
                          'missing_pct': (df_cvc.isnull().sum().values / total_cvc * 100).round(2)
                         }).sort_values('missing_pct', ascending=False)
miss_cvc.to_csv(METRICS_DIR / 'missing_values.csv', index=False)
df_cvc[numeric_cvc].apply(pd.to_numeric, errors='coerce').corr().to_csv(METRICS_DIR / 'correlation_matrix.csv')
df_cvc[numeric_cvc].describe().T.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved metrics to {METRICS_DIR}")

---
## 4. population_frequency_ml_features
**Use Case 8 — Population Carrier Screening (source table)**
**Target:** is_carrier_screening_candidate

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('population_frequency_ml_features')

print("Loading population_frequency_ml_features (full table, ~46K rows)...")
df_pf = pd.read_sql("SELECT * FROM gold.population_frequency_ml_features", engine)
print(f"Rows: {len(df_pf):,}  Cols: {len(df_pf.columns)}")

int_cols_pf = ['rarity_score', 'carrier_risk_score', 'pathogenicity_likelihood_score']
double_cols_pf = ['allele_frequency']
bool_cols_pf = [
    'is_ultra_rare_variant', 'is_very_rare_variant', 'is_rare_variant',
    'is_low_frequency_variant', 'is_common_variant',
    'is_pathogenic', 'is_benign', 'is_vus',
    'is_germline', 'is_somatic',
    'is_clinically_actionable_rare_variant', 'is_carrier_screening_candidate'
]

for col in int_cols_pf:
    if col in df_pf.columns:
        df_pf[col] = pd.to_numeric(df_pf[col], errors='coerce').astype('Int64')
for col in double_cols_pf:
    if col in df_pf.columns:
        df_pf[col] = pd.to_numeric(df_pf[col], errors='coerce')
for col in bool_cols_pf:
    if col in df_pf.columns:
        df_pf[col] = df_pf[col].astype(str).str.lower().map({'true': True, 'false': False})

total_pf = len(df_pf)

carrier_pf    = int(df_pf['is_carrier_screening_candidate'].sum())         if 'is_carrier_screening_candidate'         in df_pf.columns else 0
actionable_pf = int(df_pf['is_clinically_actionable_rare_variant'].sum())  if 'is_clinically_actionable_rare_variant'  in df_pf.columns else 0

print(f"\nTarget is_carrier_screening_candidate        : {carrier_pf:,} ({carrier_pf/total_pf*100:.1f}%)")
print(f"Target is_clinically_actionable_rare_variant : {actionable_pf:,} ({actionable_pf/total_pf*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['Carrier', 'Not Carrier'], [carrier_pf, total_pf - carrier_pf],
            color=['#f39c12', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([carrier_pf, total_pf - carrier_pf]):
    axes[0].text(i, val, f'{val:,}\n({val/total_pf*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Carrier Screening Candidate Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'frequency_tier' in df_pf.columns:
    ft_dist = df_pf['frequency_tier'].value_counts()
    ft_dist.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Frequency Tier Distribution', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variables.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variables.png")

bool_pf = [c for c in bool_cols_pf if c in df_pf.columns]
bool_rates_pf = {c: df_pf[c].sum() / total_pf * 100 for c in bool_pf}
bool_pf_df = pd.Series(bool_rates_pf).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if v > 50 else '#3498db' for v in bool_pf_df.values]
bars = ax.barh(bool_pf_df.index, bool_pf_df.values, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, bool_pf_df.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2.,
            f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Percentage (%)', fontsize=11, fontweight='bold')
ax.set_title('Boolean Feature True Rates', fontsize=12, fontweight='bold')
ax.set_xlim(0, 115)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_boolean_features.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_boolean_features.png")

if 'allele_frequency' in df_pf.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    freq_pf = df_pf['allele_frequency'].dropna()
    ax.hist(np.log10(freq_pf[freq_pf > 0] + 1e-9), bins=50,
            color='navy', alpha=0.8, edgecolor='black')
    ax.set_xlabel('log10(Allele Frequency)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title('Allele Frequency Distribution (log scale)', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '03_allele_frequency.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 03_allele_frequency.png")

miss_pf = pd.DataFrame({'column': df_pf.columns,
                         'missing_pct': (df_pf.isnull().sum().values / total_pf * 100).round(2)
                        }).sort_values('missing_pct', ascending=False)
miss_pf.to_csv(METRICS_DIR / 'missing_values.csv', index=False)
corr_f_pf = [c for c in ['rarity_score','carrier_risk_score','pathogenicity_likelihood_score','allele_frequency'] if c in df_pf.columns]
df_pf[corr_f_pf].apply(pd.to_numeric, errors='coerce').corr().to_csv(METRICS_DIR / 'correlation_matrix.csv')
df_pf[corr_f_pf].describe().T.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved metrics to {METRICS_DIR}")

## 5. Summary

In [ ]:
print("=" * 65)
print("POPULATION AND CANCER SOURCE TABLES EDA COMPLETE")
print("=" * 65)
print()
print("Tables analyzed:")
print(f"  cancer_variant_ml_features        : {total_cvc:,} rows (10% sample)")
print(f"  population_frequency_ml_features  : {total_pf:,} rows (full)")
print()
print("Target variable summary:")
print(f"  UC7/UC15 is_driver_candidate              : {driver_cvc/total_cvc*100:.1f}% positive")
print(f"  UC8 is_carrier_screening_candidate        : {carrier_pf/total_pf*100:.1f}% positive")
print(f"  UC9 is_clinically_actionable_rare_variant : {actionable_pf/total_pf*100:.1f}% positive")
print()
print("Next: 08_gene_level_tables_eda.ipynb")